# Test: böjknäckning och böjvridknäckning

Kontrollerar `kb.böjknäckning` och `kb.böjvridknäckning` mot EN 1993-1-1 6.3 och bilaga A (metod 1).

Kör med **Run All**. Kerneln ska vara `.venv` i repot.

- ✔ kontrollen gick igenom
- ✘ känt fel: fel som redan är kända och inte rättade än
- ✘ FEL: nytt fel

In [ ]:
import math
import kthbygg as kb
from kontroll import kolla, kolla_fel, klass, utskrift, sammanfattning, nollställ

nollställ()

## Eulers knäcklast

L = 5 m, E = 210 GPa, I = 4,97·10⁶ mm⁴. Fall 1 (ledad–ledad) ger N_cr = π²EI/L².

In [ ]:
kolla("Euler_Ncr fall 1", lambda: kb.böjknäckning.Euler_Ncr(5000, 210000, 4.97e6, 1), 412036.2)
kolla("Euler_Ncr fall 2 (L_cr = 2L)", lambda: kb.böjknäckning.Euler_Ncr(5000, 210000, 4.97e6, 2), 103009.1)
kolla_fel("Euler_Ncr okänt fall", lambda: kb.böjknäckning.Euler_Ncr(5000, 210000, 4.97e6, 5))

## Slankhet och reduktionsfaktor χ

Kontrollvärden: λ̄ = 1,0 med kurva b ger χ = 0,597 (tabellvärde).

In [ ]:
λ = kolla("λ_z", lambda: kb.böjknäckning.λ_z(355, 2850, 412036.2), 1.56700)
kolla("Φ_z λ̄=1,0 α=0,34", lambda: kb.böjknäckning.Φ_z(0.34, 1.0), 1.136)
kolla("χ_z λ̄=1,0 kurva b", lambda: kb.böjknäckning.χ_z(kb.böjknäckning.Φ_z(0.34, 1.0), 1.0), 0.59702)
kolla("χ_y λ̄=0,5 kurva a", lambda: kb.böjknäckning.χ_y(kb.böjknäckning.Φ_y(0.21, 0.5), 0.5), 0.92427)
kolla("χ_z λ̄=1,5 kurva d", lambda: kb.böjknäckning.χ_z(kb.böjknäckning.Φ_z(0.76, 1.5), 1.5), 0.27657)
kolla("χ ≤ 1 för liten slankhet", lambda: kb.böjknäckning.χ_y(kb.böjknäckning.Φ_y(0.34, 0.1), 0.1), 1.0)

## Momentfaktorer C_mi (tabell A.2)

In [ ]:
kolla("C_mi_1 Ψ=0, N/N_cr=0,1", lambda: kb.böjknäckning.C_mi_1(100, 1000, 0), 0.77812)
kolla("C_mi_3", lambda: kb.böjknäckning.C_mi_3(100, 1000), 0.982)
kolla("C_mi_4", lambda: kb.böjknäckning.C_mi_4(100, 1000), 1.003)
kolla_fel("C_mi_1 utan Ψ ger tydligt fel", lambda: kb.böjknäckning.C_mi_1(100, 1000), känt_fel=True)

## Interaktionsfaktorer utan vippning

k_yy = C_my / (1 − χ_y N_Ed/N_cr,y) · 1/C_yy för klass 1 och 2, utan 1/C_yy för klass 3 och 4. Indata: C_my = 0,9, χ_y = 0,6, N_Ed/N_cr = 0,25.

In [ ]:
kolla("k_yy klass 3", lambda: kb.böjknäckning.k_yy(0.9, 0.95, 500, 2000, 0.6, 3), 1.05882)
kolla("k_yy klass 1, C_yy = 0,95", lambda: kb.böjknäckning.k_yy(0.9, 0.95, 500, 2000, 0.6, 1), 1.11455, känt_fel=True)
kolla("k_zz klass 2, C_zz = 0,9", lambda: kb.böjknäckning.k_zz(0.6, 500, 2000, 0.5, 2, C_zz=0.9), 0.76190, känt_fel=True)
kolla_fel("k_yy okänd tvärsnittsklass", lambda: kb.böjknäckning.k_yy(0.9, 0.95, 500, 2000, 0.6, 7), känt_fel=True)

## C_yy och C_zz (tabell A.1)

IPE 300: W_pl,y/W_el,y = 628/557, W_pl,z/W_el,z = 125/80,5.

- C_yy med C_my = 1, λ̄_max = 1,5, n_pl = 0,5 blir 0,788 men ska vara minst W_el/W_pl = 0,887
- C_zz = 1 + (w_z − 1)[(2 − 14 C_mz² λ̄²_max / w_z⁵) n_pl] med C_mz = 0,6, λ̄_max = 0,8, n_pl = 0,3 ger 1,2363

In [ ]:
kolla("C_yy undre gräns W_el/W_pl", lambda: kb.böjknäckning.C_yy(628e3, 557e3, 1.0, 1.5, 0.5, 1.0), 0.88694, känt_fel=True)
kolla("C_zz enligt tabell A.1", lambda: kb.böjknäckning.C_zz(125e3, 80.5e3, 0.6, 0.8, 0.3, 1.0), 1.23628, känt_fel=True)

## Böjvridknäckning (tabell A.1)

C_mz = C_mz,0 och C_mLT ≥ 1.

In [ ]:
kolla("a_LT", lambda: kb.böjvridknäckning.a_LT(1e5, 1e7), 0.99)
kolla("C_mz = C_mz,0", lambda: kb.böjvridknäckning.C_mz(0.8, 0.99, 2.0), 0.8, känt_fel=True)
kolla("C_mLT ≥ 1", lambda: kb.böjvridknäckning.C_m_LT(0.9, 0.99, 100, 1000, 2000), 1.0, känt_fel=True)
kolla("k_yz_LT klass 1", lambda: kb.böjvridknäckning.k_yz_LT(0.8, 0.95, 200, 1000, 1, C_yz=0.9, w_z=1.5, w_y=1.13), 0.72969)
kolla("k_yz_LT klass 3", lambda: kb.böjvridknäckning.k_yz_LT(0.8, 0.95, 200, 1000, 3), 0.95)
kolla_fel("k_yz_LT klass 1 utan w_y, w_z", lambda: kb.böjvridknäckning.k_yz_LT(0.8, 0.95, 200, 1000, 1), känt_fel=True)

## Sammanfattning

Cellen stoppar med `AssertionError` om någon kontroll som inte är märkt som känt fel misslyckas.

In [ ]:
sammanfattning()